# Classes

## Testing a Class

In addition to testing functions, you can also use `pytest` to test classes.
By aiming for full coverage, you ensure (as much as you can) that your class is working like it should.

### A Variety of Assertions

When testing functions, we've seen that any conditional statement can be checked by `assert`. A few of the common tests are:

| Assertion | Claim |
| --- | --- |
| `assert a == b` | Assert that two values are equal |
| `assert a != b` | Assert that two values are not equal |
| `assert a` | Assert that `a` evaluates to `True` |
| `assert element in list` | Assert that an element is in a list |
| `assert element not in list` | Assert that an element is not in a list |

Anything that can be expressed as a conditional statement can be included in a test.

### A Class to Test

Let's test a class that reads in a JSON file and computes the average and RMS of the values.
A simple class `TemperatureAnalysis` is included in the module `analysis.py`.
It takes a single argument during instantiation, which is the path to a file.
There are two methods: `average()` and `rms()` that calculate the relevant value given the data in the file.

An example of its use is given below, reading in the JSON file you might recognize from your homework.

In [1]:
from analysis import TemperatureAnalysis

temp_ana = TemperatureAnalysis('weather_data.json', 'temperature_newyorkcity')
print(f'The average temperature is {temp_ana.average()} with a standard deviation of {temp_ana.std()}')

The average temperature is -5.684341886080802e-17 with a standard deviation of 3.535520499162747


### Testing the TemperatureAnalysis Class

Let's write two tests to verify that the `average()` and `std()` methods are working properly.

In [ ]:
from analysis import TemperatureAnalysis

def test_average():
    """Input file is a sine wave over 3 periods, average should be close to zero"""
    temperature_analysis = TemperatureAnalysis('weather_data.json', 'temperature_newyorkcity')
    assert abs(temperature_analysis.average()) < 1e-10


def test_std():
    """Input file is a sine wave with amplitude 5, standard deviation should be A/sqrt(2) = 3.5355"""
    temperature_analysis = TemperatureAnalysis('weather_data.json', 'temperature_newyorkcity')
    assert abs(temperature_analysis.std() - 3.5355) < 1e-4

The first thing we do is import the module that contains the class that we're testing. We then make a different test for each method we'd like to check. In each test function, we start by creating an instance of the `TemperatureAnalysis` class based on the `weather_data.json` file from homework 8. We then assert that the `average()` and `std()` functions return the expected values.

To run these, copy them into a file called `test_analysis.py` in the `12-classes/` directory, open a terminal, and run `pytest`:
```bash
cd 12-classes
pytest
```

### Using Fixtures

In testing, a *fixture* helps set up a test environment.
Often, this involves setting up a resource that is used by more than one test.
We can create a fixture in `pytest` by writing a function withe the decorator `@pytest.fixture`.
A *decorator* is a directive placed just before a function definition. Python applies this directive to the function before it runs, to alter how the function code behaves.

For now, we'll look at two types of fixtures:
1. Create a single survey instance that can be used in both test functions
2. Create a simple input file directly in the testing code

#### Creating a Single Class Instance

We'll start by creating a single survey instance that can be used in both test functions:

In [2]:
import pytest

from analysis import TemperatureAnalysis

@pytest.fixture
def temperature_analysis():
    """A temperature analysis instance that will be available to all tests"""
    temperature_analysis = TemperatureAnalysis('weather_data.json', 'temperature_newyorkcity')
    return temperature_analysis


def test_average(temperature_analysis):
    """Input file is a sine wave over 3 periods, average should be close to zero"""
    assert abs(temperature_analysis.average()) < 1e-10


def test_std(temperature_analysis):
    """Input file is a sine wave with amplitude 5, standard deviation should be A/sqrt(2) = 3.5355"""
    assert abs(temperature_analysis.std() - 3.5355) < 1e-4

ModuleNotFoundError: No module named 'pytest'

We import `pytest` now, since we're using a decorator that is defined in `pytest`.
Next, we apply the `@pytest.fixture` decorator to the new function `temperature_analysis()` that builds a `TemperatureAnalysis` object and returns it.

Notice that the definitions of both test functions have changed; each test function now has a parameter called `temperature_analysis`.
When a parameter in a test function matches the name of a function with the `@pytest.fixture` decorator, the fixture will run automatically and the return value will be passed to the test function.

While there is no new code, one line from each of the test functions has been moved to the `temperature_analysis()` function.
This makes our tests shorter, easier to write and maintain, and also easier to read.

#### Using the built-in tmp_path fixture

We created our own fixture above, however, `pytest` also includes pre-defined fixtures.
A common one is `tmp_path`, which will create a temporary path where you can write a test file that is automatically deleted after the test is run.

For example, we can use `tmp_path` as a parameter in our `temperature_analysis()` fixture. When `pytest` see's `tmp_path`, this fixture will run automatically and return a `Path` object pointing to a temporary directory.

In [ ]:
import pytest
import json

from analysis import TemperatureAnalysis

@pytest.fixture
def temperature_analysis(tmp_path):
    """A temperature analysis instance that will be available to all tests"""

    # Create a temporary JSON file with mean 0 and std 0.81649
    path = tmp_path / 'time_temp.json' # Use the '/' operator to create a path
    content = [
        {'time': 0, 'temperature': -1},
        {'time': 0, 'temperature': 0},
        {'time': 0, 'temperature': 1},
    ]
    path.write_text(json.dumps(content), encoding='utf-8')

    temperature_analysis = TemperatureAnalysis(path)
    return temperature_analysis


def test_average(temperature_analysis):
    """Test file has an average temperature of 0"""
    assert abs(temperature_analysis.average()) < 1e-10


def test_std(temperature_analysis):
    """Input file has a standard deviation of 0.81649"""
    assert abs(temperature_analysis.std() - 0.81649) < 1e-4

Because we made our own JSON file that used the default key `temperature` we didn't have to specify the `temperature_key` parameter when instantiating the class. Notice that we did need to update the two tests to check for the proper average and standard deviation of our new test file.

If your test file doesn't need to be too large, this is a nice way to package up everything needed to test your code in one place.

### Using the approx() function

The last feature we'll talk about (although there are many more) is the `approx()` function provided by `pytest`.
It allows you to assert that two numbers are equal to each other within a specified tolerance.
In our example, it would look like this:

In [ ]:
import pytest
import json

from analysis import TemperatureAnalysis

@pytest.fixture
def temperature_analysis(tmp_path):
    """A temperature analysis instance that will be available to all tests"""

    # Create a temporary JSON file with mean 0 and std 0.81649
    path = tmp_path / 'time_temp.json' # Use the '/' operator to create a path
    content = [
        {'time': 0, 'temperature': -1},
        {'time': 0, 'temperature': 0},
        {'time': 0, 'temperature': 1},
    ]
    path.write_text(json.dumps(content), encoding='utf-8')

    temperature_analysis = TemperatureAnalysis(path)
    return temperature_analysis


def test_average(temperature_analysis):
    """Test file has an average temperature of 0"""
    assert temperature_analysis.average() == pytest.approx(0.0)


def test_std(temperature_analysis):
    """Input file has a standard deviation of 0.81649"""
    assert temperature_analysis.std() == pytest.approx(0.81649)

By default, `approx()` considers two floating point numbers equal if they are within `1e-12` of each other.
It also considers them equal if their relative difference is within `1e-6`.
You can change either of these if you need too:
```python
assert temperature_analysis.average() == pytest.approx(0.0, rel=1e-4, abs=1e-10)
```